# How to Calculate SPEI 

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import gamma, norm

### Step 1: Generate Synthetic Precipitation and Potential Evapotranspiration Data


In [ ]:
np.random.seed(42)
time = np.arange(0, 120)  # 10 years of monthly data
precip = np.random.gamma(shape=2, scale=2, size=len(time))  # Synthetic precipitation
pet = np.random.uniform(1, 3, size=len(time))  # Potential Evapotranspiration

In [ ]:
deficit = precip - pet  # Climatic water balance

In [ ]:
df = xr.Dataset(
    {"deficit": ("time", deficit)}, 
    coords={"time": time}
)

### Step 2: Fit a Gamma Distribution to the Water Balance Data


In [ ]:
def fit_gamma(data):
    shape, loc, scale = gamma.fit(data, floc=0)  # Fit gamma distribution
    return shape, scale

In [ ]:
def spei_index(data, scale):
    spei = np.full_like(data, np.nan)
    for i in range(scale, len(data)):
        subset = data[i - scale:i]
        shape, scale = fit_gamma(subset)
        cdf = gamma.cdf(data[i], shape, scale=scale)
        spei[i] = norm.ppf(cdf)  # Convert to standard normal
    return spei

### Step 3: Compute SPEI Index


In [ ]:
df["SPEI"] = ("time", spei_index(df.deficit.values, scale=3))


### Step 4: Plot the Results


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df.time, df.SPEI, label="SPEI (3-month scale)")
plt.axhline(0, color="k", linestyle="--")
plt.xlabel("Time (months)")
plt.ylabel("SPEI")
plt.title("Standardized Precipitation-Evapotranspiration Index (SPEI)")
plt.legend()
plt.show()
